# 08 — Evaluation, Checkpoints, Resume, Adapters, and Merging

## What the evaluation terms mean

A **baseline** is the starting system measured under the same conditions as the trained system. **Training data** drives weight updates; **validation data** guides choices; **test data** is reserved for final assessment.

**Generalization** means useful behavior on unseen examples. **Overfitting** means the model learns training-specific patterns that do not transfer. Training loss can fall while validation behavior worsens.

**Perplexity** summarizes mean next-token negative log-likelihood. It depends on tokenizer, target masking and evaluation data; it is not answer accuracy. A **metric** is a measured summary, while a generated sample is one observed behavior.

**What problem does this solve?** Evaluation distinguishes “the optimizer ran” from “the model improved at the intended task.” Reload checks also distinguish “files exist” from “the saved artifacts reconstruct the model.” Neither a finite loss nor a successful reload alone proves useful generalization.


## NovaBot: make the comparison fair

**Fictional assessment example:**

~~~json
{"group_id":"heldout-manual","prompt":"List NovaBot's available modes.",
 "reference_answer":"idle, mapping, navigation"}
~~~

Compare base and tuned outputs with the same prompt, decoding settings, tokenizer and available context. If the same source manual supplied the training facts, answering this question may test recall or response formatting, not transfer to a new manual. State what the split actually measures.

**Hand-constructed losses:** if mean supervised-token loss is $\log 2\approx0.693$, perplexity is $\exp(\log2)=2$. If loss rises to $\log4$, perplexity is 4. Here $\log$ is the natural logarithm and $\exp$ its inverse. These are arithmetic illustrations, not run measurements.

In general,
$$\mathrm{PPL}=\exp\left(-\frac{1}{N}\sum_{t\in V}\log p_\theta(y_t\mid c_t)\right),$$
where PPL is perplexity, $V$ the scored positions, $N=|V|$, $y_t$ the target token, $c_t$ its available context, and $\theta$ the model parameters. It is the inverse geometric mean probability assigned to the scored targets, not “percentage of answers wrong.”


## What gets saved?

| Term | Meaning | Typical contents |
|---|---|---|
| Inference checkpoint | Reconstruct a model for prediction | Model config, weights, tokenizer/processor assets |
| Resumable training checkpoint | Continue an interrupted optimization process | Trainable weights, optimizer state, scheduler state, random state, data position |
| Adapter | Added parameters learned over a matching base | Low-rank matrices and explicitly saved extra modules |
| Merged model | Base matrices with adapter updates incorporated | Standalone model weights plus preprocessing assets |

**Low-Rank Adaptation (LoRA)** learns low-rank additions while freezing the base. **Quantized Low-Rank Adaptation (QLoRA)** also stores the frozen base in reduced precision. **Merge** combines an adapter update with the base; **resume** continues optimizer updates. They are different operations.

An **optimizer state** includes running quantities such as AdamW's gradient moments. A **scheduler** controls the learning-rate schedule. **Random number generator (RNG) state** determines future random draws; it is more specific than remembering the original seed. A **data cursor** records which batch comes next.

Starting again from saved weights with a fresh optimizer is a new continuation, not an exact restoration of the interrupted process.


## Connection to this notebook

baseline_loss and baseline_answer measure the starting policy. The training-state.pt file stores trainable weights, optimizer/scheduler state, Python/NumPy/PyTorch RNG states, and next_batch at an optimizer-update boundary.

The experiment performs a second update uninterrupted, restores the earlier state, repeats that same second update, and compares parameters and optimizer/scheduler state. Exact equality here is a controlled same-environment CPU check, not a promise across hardware or distributed layouts.

save_pretrained() writes inference artifacts. The reload cells reconstruct a separate model and processor, then compare deterministic **greedy decoding**, which chooses the highest-probability next token. The merger reloads an unquantized base before incorporating adapter weights. Switching from quantized to unquantized inference can change numerical outputs, so the merge comparison uses an unquantized base-plus-adapter on both sides.

The evaluation workflow explicitly records the checkpoint under assessment. Its default is the run's final artifact; a baseline requires an explicit checkpoint selection. Teacher/student agreement, preference accuracy, reward and token loss are different metrics—do not exponentiate an arbitrary reinforcement or preference loss and call it perplexity.


## Common confusions and quick check

Repeatedly choosing settings from the test set makes it part of model selection. Identical greedy output on one prompt also does not prove equality on every possible input.

1. Why can two runs restart from identical weights and then diverge after one AdamW step?
2. Does a lower perplexity guarantee that NovaBot's mode count is factually correct?

<details>
<summary>Answers</summary>

1. Their optimizer moments, scheduler position, random state or next data batch may differ. Exact resume restores those states as well as weights.
2. No. Perplexity scores reference-token likelihood under a particular dataset and mask. Fact correctness needs a suitable task-level assessment.

</details>


## Before running the experiment

**Learning goals:** measure baselines, distinguish inference artifacts from optimizer checkpoints, verify deterministic resume, reload adapters, and merge an unquantized base.

Run in a fresh kernel. All weights and files are local. Tiny random checkpoints demonstrate mechanics, not model quality. [Course index](README.md) · [Flow diagrams](../docs/EXECUTION_AND_DATA_FLOW.md)

[Terminology reference](../docs/GLOSSARY.md) · [Compare training methods](../docs/TRAINING_METHODS.md)


## Experiment parameters

For local weights, set `FTLAB_DEVICE` to `cpu`, `mps`, `cuda`, or `auto` before launching. CPU/MPS default to ordinary LoRA; CUDA retains its QLoRA profile. Restart the kernel when changing devices after a Trainer has initialized. Selecting a device does not guarantee the full experiment fits its memory.


In [ ]:
LESSON = "08"
# Parameters: change these before running the notebook from top to bottom.
import copy
import csv
import json
import math
import os
import random
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

from finetunelab.devices import (
    activate_runtime,
    device_rng_state,
    empty_device_cache,
    resolve_runtime,
    restore_device_rng_state,
)
from finetunelab.education import (
    inspect_local_checkpoint,
    make_tiny_checkpoint,
    project_root,
    token_table,
)
from finetunelab.tuning import parameter_report

MODE = os.environ.get("FTLAB_NOTEBOOK_MODE", "tiny_cpu")
LOCAL_MODEL_PATH = Path(os.environ.get("FTLAB_LOCAL_MODEL", "models/Qwen3.5-2B"))
LOCAL_TEACHER_PATH = Path(os.environ.get("FTLAB_LOCAL_TEACHER", "models/Qwen3.5-4B"))
ROOT = project_root()
DATA_ROOT = Path(os.environ.get("FTLAB_LESSON_DATA", str(ROOT / "examples/education")))
OUTPUT_ROOT = Path(os.environ.get("FTLAB_NOTEBOOK_OUTPUT", str(ROOT / "outputs/notebooks")))
OUTPUT = OUTPUT_ROOT / LESSON
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
assert MODE in {"tiny_cpu", "local_pretrained"}
# tiny_cpu remains an offline CPU fixture; real checkpoints use the selected backend.
REQUESTED_DEVICE = "cpu" if MODE == "tiny_cpu" else os.environ.get("FTLAB_DEVICE", "auto")
RUNTIME = resolve_runtime(device=REQUESTED_DEVICE, dtype=os.environ.get("FTLAB_DTYPE"))
activate_runtime(RUNTIME)
DEVICE = torch.device(RUNTIME.device)
DTYPE = RUNTIME.torch_dtype
ATTENTION = "eager" if MODE == "tiny_cpu" else RUNTIME.attention
print(RUNTIME.report())
MODEL_PATH = (
    make_tiny_checkpoint(OUTPUT / "initial", seed=SEED)
    if MODE == "tiny_cpu"
    else LOCAL_MODEL_PATH.expanduser().resolve()
)
checkpoint_info = inspect_local_checkpoint(MODEL_PATH)
print({"mode": MODE, "device": str(DEVICE), "checkpoint": str(MODEL_PATH)})
print(checkpoint_info["files"])
if DEVICE.type == "mps":
    torch.mps.manual_seed(SEED)

## Load model and processor


In [ ]:
# A checkpoint includes both weights and the preprocessing contract.
processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
tokenizer = processor.tokenizer
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
load_kwargs = {
    "local_files_only": True,
    "dtype": DTYPE,
    "device_map": {"": str(DEVICE)},
    "attn_implementation": ATTENTION,
}
# Quantization is a separate choice. CPU/MPS use unquantized LoRA.
_default_qlora = (
    MODE == "local_pretrained" and DEVICE.type == "cuda" and LESSON not in {"00a", "00b", "04"}
)
USE_QLORA = os.environ.get("FTLAB_USE_QLORA", str(_default_qlora)).lower() in {"1", "true", "yes"}
if USE_QLORA and DEVICE.type != "cuda":
    raise ValueError("This CPU/MPS profile supports ordinary LoRA; set FTLAB_USE_QLORA=false.")
if USE_QLORA:
    from transformers import BitsAndBytesConfig

    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    load_kwargs["device_map"] = {"": torch.cuda.current_device()}
model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
if not USE_QLORA:
    model.to(DEVICE)
model.config.use_cache = False
print(type(model).__name__, parameter_report(model))

## From local Q&A to train/validation/test records

A dataset row is not yet a tensor. Preserve the original group identity so examples from one conversation stay together. These tiny held-out splits demonstrate plumbing; use representative, larger splits in real experiments.


In [ ]:
# Convert local Q&A rows to canonical conversations; preserve provenance.
QA_FILE = Path(os.environ.get("FTLAB_QA_FILE", str(DATA_ROOT / "qa.csv")))
if QA_FILE.suffix.lower() == ".csv":
    with QA_FILE.open(encoding="utf-8", newline="") as handle:
        raw_rows = list(csv.DictReader(handle))
elif QA_FILE.suffix.lower() == ".jsonl":
    raw_rows = [
        json.loads(line)
        for line in QA_FILE.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
else:
    raise ValueError("This converter accepts CSV or JSONL Q&A files.")
for row in raw_rows:
    if (
        not row.get("group_id")
        or not row.get("question", "").strip()
        or not row.get("answer", "").strip()
    ):
        raise ValueError("Every Q&A needs a group_id, nonempty question, and nonempty answer.")
    row.setdefault("rejected", "")
records = [
    {
        "group_id": row["group_id"],
        "messages": [
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["answer"]},
        ],
        "prompt": row["question"],
        "chosen": row["answer"],
        "rejected": row["rejected"],
    }
    for row in raw_rows
]


def split_records(rows, seed=SEED):
    # Deduplicate before splitting. Never split one document/conversation group.
    unique = {}
    for row in rows:
        key = json.dumps(row["messages"], sort_keys=True, ensure_ascii=False)
        unique.setdefault(key, row)
    groups = sorted({row["group_id"] for row in unique.values()})
    if len(groups) < 3:
        raise ValueError("Provide at least three independent document/conversation groups.")
    random.Random(seed).shuffle(groups)
    validation_groups, test_groups = set(groups[:1]), set(groups[1:2])
    splits = {"train": [], "validation": [], "test": []}
    for row in unique.values():
        split = (
            "validation"
            if row["group_id"] in validation_groups
            else "test"
            if row["group_id"] in test_groups
            else "train"
        )
        splits[split].append(row)
    return splits


splits = split_records(records)
for split, rows in splits.items():
    with (OUTPUT / f"{split}.jsonl").open("w", encoding="utf-8") as handle:
        for row in rows:
            canonical = {"group_id": row["group_id"], "messages": row["messages"]}
            handle.write(json.dumps(canonical, ensure_ascii=False) + "\n")
print({split: len(rows) for split, rows in splits.items()})
print("Raw:", raw_rows[0])
print("Canonical:", records[0])

## Chat rendering, tokenization and loss masking

The chat template supplies role delimiters. Attention masks describe real positions versus padding; labels choose prediction targets. A user token can be visible to attention while its label is `-100`. Assistant EOS should remain supervised even if its ID equals the padding ID.


In [ ]:
import re


def render(messages, generation=False):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=generation,
        enable_thinking=False,
    )


def encode_conversation(messages):
    # Template-provided generation masks are preferred. They include assistant EOS.
    template = tokenizer.chat_template or ""
    if re.search(r"{%-?\s*generation\s*-?%}", template):
        encoded = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_assistant_tokens_mask=True,
            enable_thinking=False,
        )
        ids = encoded["input_ids"]
        supervised = encoded["assistant_masks"]
    else:
        # For templates without generation annotations, verify prefix alignment.
        # Do not guess a token count by separately tokenizing the answer.
        ids = tokenizer(render(messages), add_special_tokens=False)["input_ids"]
        supervised = [0] * len(ids)
        for index, message in enumerate(messages):
            if message["role"] != "assistant":
                continue
            prefix = tokenizer(render(messages[:index], generation=True), add_special_tokens=False)[
                "input_ids"
            ]
            completed = tokenizer(render(messages[: index + 1]), add_special_tokens=False)[
                "input_ids"
            ]
            if ids[: len(prefix)] != prefix or ids[: len(completed)] != completed:
                raise ValueError(
                    "Template is not prefix-stable; use a training template with generation tags."
                )
            supervised[len(prefix) : len(completed)] = [1] * (len(completed) - len(prefix))
    if not any(supervised[1:]):
        raise ValueError("No assistant target tokens remain.")
    return {
        "input_ids": ids,
        "labels": [t if keep else -100 for t, keep in zip(ids, supervised, strict=False)],
    }


def collate_text(rows):
    items = [encode_conversation(row["messages"]) for row in rows]
    encoded = tokenizer.pad(
        [{"input_ids": item["input_ids"]} for item in items],
        padding=True,
        return_tensors="pt",
    )
    # Padding labels are independent of the pad token ID (pad may equal EOS).
    labels = torch.full_like(encoded["input_ids"], -100)
    for index, item in enumerate(items):
        labels[index, : len(item["labels"])] = torch.tensor(item["labels"])
    encoded["labels"] = labels
    return dict(encoded)


batch = collate_text(splits["train"][:2])
print(render(splits["train"][0]["messages"]))
print({name: tuple(value.shape) for name, value in batch.items()})
display(token_table(tokenizer, batch))

## Select parameters that may change

LoRA freezes the pretrained matrices and adds low-rank updates. Its zero-initialized B matrices mean some A gradients may be zero on the first step; at least one adapter must change. The frozen vision backbone is excluded. Set `TUNING` to `full` or `selective` to compare on the tiny model; full tuning of a real model needs substantially more memory.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# This is the same choice represented by tuning.strategy in a framework YAML.
TUNING = "lora"
if USE_QLORA and TUNING != "lora":
    raise ValueError("Full/selective tuning requires reloading with USE_QLORA=False.")
if USE_QLORA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()
if TUNING == "lora":
    model = get_peft_model(
        model,
        LoraConfig(
            r=int(os.environ.get("FTLAB_LORA_RANK", "4" if MODE == "tiny_cpu" else "16")),
            lora_alpha=8 if MODE == "tiny_cpu" else 32,
            target_modules="all-linear",
            exclude_modules=r".*(?:visual|vision).*",
            task_type="CAUSAL_LM",
            lora_dropout=0.0,
        ),
    )
elif TUNING == "selective":
    for name, parameter in model.named_parameters():
        parameter.requires_grad = ("norm" in name or "lm_head" in name) and "visual" not in name
elif TUNING != "full":
    raise ValueError(TUNING)
if MODE == "local_pretrained" and not USE_QLORA:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
trainable = [p for p in model.parameters() if p.requires_grad]
print(parameter_report(model))
# A small parameter sample avoids copying a 2B/4B model just to audit updates.
before = {name: p.detach().flatten()[:32].cpu().clone() for name, p in model.named_parameters()}
frozen_names = {name for name, p in model.named_parameters() if not p.requires_grad}

## Establish a baseline

Use `eval()` plus `no_grad()` for measurement, and switch back to `train()` for updates. We aggregate causal loss by supervised token count. Greedy decoding makes the before/after and reload comparisons reproducible on the same device.


In [ ]:
def to_device(batch):
    return {key: value.to(DEVICE) for key, value in batch.items()}


def precision_context():
    return RUNTIME.precision_context()


def validation_loss(current_model, rows, collator=collate_text):
    current_model.eval()
    weighted_loss, target_count = 0.0, 0
    with torch.no_grad(), precision_context():
        for row in rows:
            encoded = to_device(collator([row]))
            count = int((encoded["labels"][:, 1:] != -100).sum())
            loss = current_model(**encoded, use_cache=False).loss
            weighted_loss += float(loss) * count
            target_count += count
    return weighted_loss / max(target_count, 1)


def generate_answer(current_model, prompt="What is the color of sky ?"):
    current_model.eval()
    text = render([{"role": "user", "content": prompt}], generation=True)
    inputs = tokenizer(text, add_special_tokens=False, return_tensors="pt")
    inputs.pop("token_type_ids", None)
    with torch.no_grad(), precision_context():
        tokens = current_model.generate(
            **to_device(dict(inputs)),
            max_new_tokens=4 if MODE == "tiny_cpu" else 32,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokens[:, inputs["input_ids"].shape[1] :].cpu()


baseline_loss = validation_loss(model, splits["validation"])
baseline_answer = generate_answer(model)
print(
    {
        "baseline_validation_loss": baseline_loss,
        "baseline_answer": tokenizer.decode(baseline_answer[0], skip_special_tokens=True),
    }
)

## A resumable checkpoint is larger than an adapter

Save at an optimizer-step boundary. To resume exactly, restore trainable weights, optimizer moments, scheduler position, RNG states and the next data position. These short batches are fixed; a shuffled loader also needs its sampler/generator state. In distributed training use Accelerate/Trainer state APIs per rank.


In [ ]:
from peft import get_peft_model_state_dict, set_peft_model_state_dict

optimizer = torch.optim.AdamW(trainable, lr=1e-3)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: 1 / (step + 1))
batches = [collate_text(splits["train"][i : i + 2]) for i in range(0, 4, 2)]
global_step = 0


def update(batch):
    global global_step
    model.train()
    optimizer.zero_grad(set_to_none=True)
    with precision_context():
        loss = model(**to_device(batch), use_cache=False).loss
    assert torch.isfinite(loss)
    loss.backward()
    assert all(torch.isfinite(p.grad).all() for p in trainable if p.grad is not None)
    torch.nn.utils.clip_grad_norm_(trainable, 1.0)
    optimizer.step()
    scheduler.step()
    global_step += 1
    return float(loss.detach())


first_loss = update(batches[0])
checkpoint_path = OUTPUT / "training-state.pt"
is_adapter = hasattr(model, "peft_config")
weights = get_peft_model_state_dict(model) if is_adapter else model.state_dict()
torch.save(
    {
        "model": weights,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "global_step": global_step,
        "next_batch": 1,
        "torch_rng": torch.get_rng_state(),
        "device_rng": device_rng_state(DEVICE),
        "python_rng": random.getstate(),
        "numpy_rng": (
            np.random.get_state()[0],
            np.random.get_state()[1].tolist(),
            *np.random.get_state()[2:],
        ),
    },
    checkpoint_path,
)
# Continue uninterrupted, then remember its trainable result.
continuous_loss = update(batches[1])
continuous_parameters = {
    name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad
}
continuous_optimizer = copy.deepcopy(optimizer.state_dict())
continuous_scheduler = copy.deepcopy(scheduler.state_dict())
# Restore exactly the saved step. Only load files created by this experiment.
state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
if is_adapter:
    set_peft_model_state_dict(model, state["model"])
else:
    model.load_state_dict(state["model"])
optimizer.load_state_dict(state["optimizer"])
scheduler.load_state_dict(state["scheduler"])
global_step = state["global_step"]
torch.set_rng_state(state["torch_rng"].cpu())
restore_device_rng_state(DEVICE, state.get("device_rng"))
random.setstate(state["python_rng"])
numpy_state = state["numpy_rng"]
np.random.set_state((numpy_state[0], np.array(numpy_state[1], dtype=np.uint32), *numpy_state[2:]))
resumed_loss = update(batches[state["next_batch"]])
assert global_step == 2 and scheduler.state_dict() == continuous_scheduler
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        torch.testing.assert_close(parameter.cpu(), continuous_parameters[name], rtol=0, atol=0)
for index, expected in continuous_optimizer["state"].items():
    for key, value in expected.items():
        actual = optimizer.state_dict()["state"][index][key]
        if isinstance(value, torch.Tensor):
            torch.testing.assert_close(actual, value, rtol=0, atol=0)
        else:
            assert actual == value
assert continuous_loss == resumed_loss
print(
    {
        "uninterrupted_loss": continuous_loss,
        "resumed_loss": resumed_loss,
        "global_step": global_step,
        "optimizer_and_scheduler_match": True,
    }
)

## Inspect updates and held-out behavior

A finite loss and a changed adapter prove an update occurred, not that a model became useful. Inspect validation loss and example generations together. Keep the test set out of hyperparameter selection.


In [ ]:
changed = []
for name, parameter in model.named_parameters():
    same = torch.equal(before[name], parameter.detach().flatten()[:32].cpu())
    if name in frozen_names:
        assert same, f"Frozen parameter changed: {name}"
    elif not same:
        changed.append(name)
assert changed, "No trainable parameter sample changed."
after_loss = validation_loss(model, splits["validation"])
after_answer = generate_answer(model)
test_loss = validation_loss(model, splits["test"])
print(
    {
        "baseline_loss": baseline_loss,
        "after_loss": after_loss,
        "test_loss": test_loss,
        "changed_parameter_samples": changed[:5],
    }
)
print("Before:", tokenizer.decode(baseline_answer[0], skip_special_tokens=True))
print("After: ", tokenizer.decode(after_answer[0], skip_special_tokens=True))
# Do not assert that generalization improves after one synthetic update.
assert math.isfinite(after_loss) and math.isfinite(test_loss)

## Save, reload, and verify

`save_pretrained()` saves inference artifacts; it does not save the optimizer or training position. A PEFT artifact needs its original base. This cell creates a separate model object and compares generated token IDs. Lesson 08 covers exact resume and merging.


In [ ]:
from peft import PeftModel

artifact = OUTPUT / "final"
model.save_pretrained(artifact, safe_serialization=True)
processor.save_pretrained(artifact)
expected_tokens = generate_answer(model)
reloaded_processor = AutoProcessor.from_pretrained(artifact, local_files_only=True)
assert reloaded_processor.tokenizer.get_vocab() == tokenizer.get_vocab()
assert reloaded_processor.chat_template == processor.chat_template
processor = reloaded_processor
tokenizer = processor.tokenizer
# Release training models, optimizer state and graph references before independent reload.
for _name in (
    "model",
    "optimizer",
    "scheduler",
    "trainable",
    "parameter",
    "p",
    "outputs",
    "loss",
    "reference",
    "teacher",
    "reward_model",
    "value_model",
    "policy_optimizer",
    "value_optimizer",
    "reward_optimizer",
    "weights",
    "state",
    "logits",
    "student_logits",
    "teacher_logits",
    "new_token_logp",
    "new_logp",
    "ratio",
    "policy_loss",
    "value_loss",
    "reward_loss",
    "margin",
    "per_token_divergence",
):
    globals().pop(_name, None)
empty_device_cache(DEVICE)

# Reload independently, rather than reusing the trained Python object.
if (artifact / "adapter_config.json").exists():
    reload_base = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
    if not USE_QLORA:
        reload_base.to(DEVICE)
    reloaded = PeftModel.from_pretrained(reload_base, artifact, local_files_only=True)
else:
    reloaded = AutoModelForImageTextToText.from_pretrained(artifact, **load_kwargs)
    if not USE_QLORA:
        reloaded.to(DEVICE)
actual_tokens = generate_answer(reloaded)
assert torch.equal(expected_tokens, actual_tokens), "Greedy outputs changed after reload."
report = {
    "mode": MODE,
    "runtime": RUNTIME.report(),
    "base_checkpoint": str(MODEL_PATH),
    "artifact": str(artifact),
    "baseline_validation_loss": baseline_loss,
    "validation_loss": after_loss,
    "test_loss": test_loss,
    "reload_tokens_equal": True,
}
(OUTPUT / "lesson_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)
del reloaded
if "reload_base" in globals():
    del reload_base
empty_device_cache(DEVICE)

## Interpretation, common failures, and exercises

- If loss is NaN, inspect the number of supervised targets, precision and learning rate before adding steps.
- If every label is `-100`, repair the template/mask; an empty objective cannot teach anything.
- If frozen parameters change, inspect the trainable parameter report and optimizer parameter list.
- If tiny generations look meaningless, that is expected from random initialization and a tiny vocabulary.

**Exercises:** (1) Print which token predicts the first answer token. (2) Compare full/selective/LoRA parameter counts. (3) Change one training answer, rerun from the same seed, and inspect held-out loss. (4) Explain why saving an adapter is not enough to resume AdamW.

**Expected result:** finite objective values, some expected trainable weights changed, frozen weights unchanged, and identical greedy tokens after reload. Record actual values in `lesson_report.json`; no fixed quality threshold is asserted.


## Merge with an unquantized base

Merging adds low-rank updates to the base matrices. Reload the original base without NF4 before merging. Verify against that unquantized base-plus-adapter, since switching from quantized inference can change numerical results. A merged artifact no longer needs PEFT to load.


In [ ]:
from peft import PeftModel

merge_base = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    dtype=DTYPE,
    attn_implementation=ATTENTION,
).to(DEVICE)
merge_adapter = PeftModel.from_pretrained(
    merge_base, OUTPUT / "final", local_files_only=True
).eval()
before_merge = generate_answer(merge_adapter)
merged = merge_adapter.merge_and_unload()
after_merge = generate_answer(merged)
if MODE == "tiny_cpu":
    assert torch.equal(before_merge, after_merge)
merged.save_pretrained(OUTPUT / "merged")
processor.save_pretrained(OUTPUT / "merged")
print(
    {
        "merge_tokens_equal": torch.equal(before_merge, after_merge),
        "merged_artifact": str(OUTPUT / "merged"),
    }
)
del merged, merge_adapter, merge_base
reloaded_merged = AutoModelForImageTextToText.from_pretrained(
    OUTPUT / "merged",
    local_files_only=True,
    dtype=DTYPE,
    attn_implementation=ATTENTION,
).to(DEVICE)
assert torch.equal(after_merge, generate_answer(reloaded_merged))
report["merged_reload_tokens_equal"] = True
report["resume_parameters_optimizer_scheduler_equal"] = True
(OUTPUT / "lesson_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
del reloaded_merged

## Evaluate the saved artifact through the CLI workflow

`evaluate` defaults to `output_dir/final`; it must fail if that artifact does not exist. To measure the base, pass its checkpoint explicitly. Evaluation writes into a separate directory and records the actual checkpoint, so it does not overwrite the training configuration.


In [ ]:
from finetunelab.config import RECIPE_ADAPTER
from finetunelab.workflows import evaluate

config = RECIPE_ADAPTER.validate_python(
    {
        "method": "sft",
        "model": {
            "name_or_path": str(MODEL_PATH),
            "dtype": RUNTIME.dtype,
            "local_files_only": True,
        },
        "data": {
            "source": str(OUTPUT),
            "source_type": "local",
            "data_files": {name: f"{name}.jsonl" for name in splits},
            "eval_split": "validation",
            "max_length": 128,
        },
        "tuning": {"strategy": "lora"},
        "training": {
            "output_dir": str(OUTPUT),
            "device": RUNTIME.device,
            "bf16": RUNTIME.bf16,
            "fp16": RUNTIME.fp16,
            "tf32": False,
            "gradient_checkpointing": False,
            "report_to": ["none"],
        },
        "evaluation": {"max_samples": 1, "generate_samples": 1},
        "generation": {"max_new_tokens": 4},
    }
)
if MODE == "tiny_cpu":
    artifact_metrics = evaluate(config)
    base_metrics = evaluate(config, checkpoint=str(MODEL_PATH))
    assert artifact_metrics["checkpoint"] == str(OUTPUT / "final")
    assert base_metrics["checkpoint"] == str(MODEL_PATH)
    print({"artifact": artifact_metrics, "base": base_metrics})
else:
    print("Run ftlab evaluate with this configuration in a fresh process.")
print("Artifacts:", sorted(path.name for path in OUTPUT.iterdir()))

## Exercises and interpretation

1. Remove optimizer state from a copy of the checkpoint and explain why the continuation changes.
2. Use a new held-out prompt and compare base, adapter and merged generation.
3. Find the evaluated checkpoint identity in the evaluation manifest.
4. Compare tokens as well as floating-point logits when changing dtype or quantization.

Exact CPU continuation verifies saved training state. It does not guarantee bitwise equality across different CUDA kernels, distributed world sizes or library versions.
